# Demo 02: Grounded Answers You Can Trust

Week 4, Module 01. Reliability, hallucination risk, and grounding.

Instructor demo (we-do). We build this together, live. Where you see a WE-DO banner in a cell, that is a line we write as a room. Everything else is pre-filled so we spend our time on the ideas, not the plumbing.

## The retail hook

A Cordwell Home and Hardware associate asks an internal assistant a policy question: what is the return window, how much torque does a drill have, is a product real. A confident wrong answer here is worse than no answer, because an associate will repeat it to a customer. So we build the assistant to do three things: pull evidence from a knowledge base, cite the exact source next to every claim, and decline when the evidence is weak. Then we measure how well it stayed true to its sources.

## Engineer framing

Two ideas map onto tools you already trust.

- Retrieval is a search index over your docs. We rank chunks by similarity to the question, the same shape as ranking rows by a score.
- A faithfulness metric is a unit test for grounding. It checks that each answer sentence actually matches the source it cites, instead of trusting the model to be honest.

## What you will be able to do

1. Build a small retrieval index over a document set and rank chunks by similarity.
2. Synthesize an extractive answer that cites its sources inline as [doc_id:chunk_id].
3. Add an abstention policy that declines when the top match is too weak.
4. Compute faithfulness proxies: attribution precision, evidence sufficiency, and a grounding score.
5. Show three failure modes a grounded system handles: a conflicting source, a fabricated product, and an off-topic question.

Time budget: about 50 minutes. Natural break after the charts in Part 6. The generative section in Part 8 is optional.

## Worked target output

This is what we are building toward. For the question "What is the return window for most items?", the finished assistant produces an answer where every sentence carries a citation, plus a small set of faithfulness scores. Verified output from this notebook:

```
QUERY: What is the return window for most items?
best chunk similarity: 0.264
answer: Most items may be returned within 90 days with proof of purchase. [returns_policy:0]
        Major appliances have a 48 hour window for reporting cosmetic damage. [returns_policy:0]
faithfulness: AP=1.00  ES=0.69  GS=1.00
```

AP is attribution precision, ES is evidence sufficiency, GS is grounding score. We define all three in Part 5. Seeing the target shape now means you code toward a visible goal, not toward a failing assertion.

## Part 0: Setup

No GPU and no model download here. This whole demo runs on scikit-learn and numpy, so it starts instantly on the cohort Macs. The only optional piece is a local language model in Part 8, and that stays off by default.

We print versions so a mismatch is visible up front.

In [ ]:
# If the stack is missing, uncomment:
# %pip install -q scikit-learn numpy pandas matplotlib

import os
import re
import sys
import platform
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
import sklearn
print("scikit-learn:", sklearn.__version__)

## Part 1: The knowledge base

A small, purpose-built Cordwell knowledge base. Eight short synthetic documents. Most are ordinary policy and product content. Three carry deliberate traps we will use later: an official drill spec with a source tag, a conflicting forum claim about the same drill, and a note about a product that does not exist. All content is fictional.

Point out that in production you would point this at a folder of real policy documents. The pipeline does not change.

In [ ]:
DOCS = {
    "returns_policy": (
        "Most items may be returned within 90 days with proof of purchase. "
        "Major appliances have a 48 hour window for reporting cosmetic damage. "
        "Outdoor power equipment must be unused and in original packaging to qualify."
    ),
    "delivery": (
        "Standard home delivery takes 3 to 5 business days after the order ships. "
        "Appliance haul away of an old unit is available for an added fee. "
        "The delivery window is confirmed by text message the day before arrival."
    ),
    "warranty": (
        "Cordwell branded power tools carry a 3 year limited warranty. "
        "An extended protection plan adds 2 more years of coverage. "
        "A warranty claim requires the receipt and the model number."
    ),
    "install_service": (
        "Dishwasher installation includes leveling, the water line hookup, and old unit removal. "
        "Installation is scheduled separately from delivery. "
        "An associate confirms the install appointment by phone."
    ),
    "drill_spec_official": (
        "The Timberline 18 volt drill delivers 500 inch pounds of torque per the product sheet. "
        "It ships with one battery and a charger. [source: product-sheet-TL18V]"
    ),
    "drill_forum_conflicting": (
        "An unofficial forum post claims the Timberline 18 volt drill delivers 900 inch pounds of torque. "
        "This conflicts with the official product sheet and is not a verified source."
    ),
    "fabricated_product": (
        "Associates are sometimes asked about the Cordwell SkyHammer 9000, which is not a real catalog product. "
        "Requests for a stock number that does not exist should be declined, not guessed."
    ),
    "abstention_policy": (
        "When the knowledge base lacks a confident answer the assistant declines and routes to a human associate. "
        "For policy and safety questions, declining is preferred over a confident wrong answer."
    ),
}
print(f"{len(DOCS)} documents loaded.")

## Part 2: Chunking with overlap and dedup hygiene

Retrieval works on chunks, not whole documents, so a citation can point at a precise span. We split each document into short overlapping windows of sentences. Overlap reduces the chance that a fact gets split across a boundary and lost. Then we drop near duplicate chunks with a cheap character shingle comparison, so the same text does not crowd out variety in the results.

This cell is plumbing. It is pre-written for both of us. The concepts to name out loud are overlap and dedup, not the string handling.

In [ ]:
@dataclass
class Chunk:
    doc_id: str
    chunk_id: int
    text: str

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

def make_chunks(docs, window=2, overlap=1):
    """Overlapping sentence windows. window is sentences per chunk, overlap is shared sentences."""
    chunks = []
    step = max(1, window - overlap)
    for doc_id, text in docs.items():
        sents = split_sentences(text)
        cid = 0
        for i in range(0, len(sents), step):
            win = sents[i:i + window]
            if not win:
                break
            chunks.append(Chunk(doc_id, cid, " ".join(win)))
            cid += 1
            if i + window >= len(sents):
                break
    return chunks

def shingles(s, k=7):
    s = re.sub(r"\s+", " ", s.lower()).strip()
    return {s[i:i + k] for i in range(max(1, len(s) - k + 1))}

def dedup_chunks(chunks, threshold=0.9):
    """Drop a chunk if its character shingles overlap an earlier chunk above threshold (Jaccard)."""
    kept, sigs = [], []
    for ch in chunks:
        sh = shingles(ch.text)
        is_dup = any(len(sh & prev) / (len(sh | prev) or 1) >= threshold for prev in sigs)
        if not is_dup:
            kept.append(ch)
            sigs.append(sh)
    return kept

chunks = dedup_chunks(make_chunks(DOCS))
print(f"{len(chunks)} chunks after dedup.")
for c in chunks[:4]:
    print(f"  {c.doc_id}:{c.chunk_id}  {c.text[:60]}...")

## Part 3: Build the retrieval index

We turn each chunk into a TF-IDF vector and rank chunks against a query with cosine similarity. TF-IDF is a sparse, transparent baseline. It is enough to teach grounding, citations, and faithfulness without a heavyweight embedding model, and it runs instantly. In Week 5 you swap this for dense embeddings; the surrounding pipeline is unchanged.

WE-DO here. The two lines that are the whole point of retrieval: score every chunk against the query, then take the top k.

In [ ]:
CHUNK_TEXTS = [c.text for c in chunks]
CHUNK_IDS = [(c.doc_id, c.chunk_id) for c in chunks]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")
X = vectorizer.fit_transform(CHUNK_TEXTS)

def retrieve(query, k=4):
    """Rank chunks by cosine similarity to the query. Returns (score, (doc_id, chunk_id), text)."""
    qv = vectorizer.transform([query])
    # ===================== WE-DO =====================
    # 1. sims: cosine similarity of the query vector qv against every chunk row in X,
    #    flattened to a 1D array with .ravel().
    # 2. idx: the indices of the top k chunks, highest score first. Sorting -sims
    #    ascending gives highest first. Keep the first k.
    sims = ...          # replace ... together
    idx = ...           # replace ... together
    # =================================================
    return [(float(sims[i]), CHUNK_IDS[i], CHUNK_TEXTS[i]) for i in idx]

print(f"index built over {X.shape[0]} chunks, {X.shape[1]} features")

Quick smoke test. Run the retriever on one question and read the ranked chunks. This is the target shape for retrieval: a score, a source id, and the chunk text, ordered from best to worst.

In [ ]:
for score, (doc_id, cid), text in retrieve("What is the return window?", k=3):
    print(f"  {score:.3f}  {doc_id}:{cid}  {text[:60]}...")

## Part 4: Answer synthesis with citations and abstention

Now we turn ranked chunks into an answer. We keep it extractive on purpose: we select the sentences most relevant to the question and attach the citation of the chunk each one came from. Extractive answers are faithful by construction, because every sentence is literally copied from a cited source. That gives us a clean baseline to measure against, and later a contrast with a generative model that might embellish.

Two design choices to name.

- Abstention. If the best chunk similarity is below a threshold, we return INSUFFICIENT_INFORMATION instead of an answer. Declining on weak evidence is the single most important reliability behavior in this demo.
- Dedup at the sentence level. Overlapping chunks can surface the same sentence twice. We keep the first occurrence so the answer does not repeat itself.

WE-DO here. Two lines. The abstention gate, and the line that attaches a citation to a sentence.

In [ ]:
SIM_THRESHOLD = 0.15   # tuned so off-topic questions fall below it and abstain

def synthesize(query, k=4, threshold=SIM_THRESHOLD, top_sentences=2):
    retrieved = retrieve(query, k=k)
    best = retrieved[0][0] if retrieved else 0.0
    # ===================== WE-DO =====================
    # Abstention gate: if the best chunk similarity is below threshold, decline.
    # Return the INSUFFICIENT_INFORMATION dict below instead of building an answer.
    if ...:        # replace ... together
        return {"answer": "INSUFFICIENT_INFORMATION", "cited_ids": [], "retrieved": retrieved, "best_sim": best}
    # =================================================

    qv = vectorizer.transform([query])
    cand = []
    for score, (doc_id, cid), text in retrieved:
        for s in split_sentences(text):
            cand.append((s, doc_id, cid))
    rel = cosine_similarity(qv, vectorizer.transform([c[0] for c in cand])).ravel()

    lines, cited, seen = [], [], set()
    for j in np.argsort(-rel):
        s, doc_id, cid = cand[j]
        key = re.sub(r"\s+", " ", s.lower()).strip()
        if key in seen:
            continue
        seen.add(key)
        # ===================== WE-DO =====================
        # Append the sentence with its source citation attached, in the form
        # "sentence text [doc_id:chunk_id]". Use an f-string over s, doc_id, cid.
        lines.append(...)        # replace ... together
        # =================================================
        cited.append((doc_id, cid))
        if len(lines) >= top_sentences:
            break
    return {"answer": " ".join(lines), "cited_ids": cited, "retrieved": retrieved, "best_sim": best}

out = synthesize("What is the return window for most items?")
print("best_sim:", round(out["best_sim"], 3))
print("answer:", out["answer"])

## Part 5: Faithfulness metrics

Three lightweight proxies, computed with the same TF-IDF vectors. They are proxies, not gold labels, and we say so out loud. For true faithfulness you use human review or a natural language inference verifier. These are fast, transparent checks suitable for a live demo and for a first automated gate.

- Attribution precision (AP). Fraction of answer sentences that carry a valid citation. In this extractive pipeline we attach one per sentence, so AP is 1.00 by construction. That is the point of the design, not a bug, and it is worth stating plainly rather than dressing it up.
- Evidence sufficiency (ES). For each answer sentence, cosine similarity between the sentence and the chunk it cites. This is the interesting one. It asks whether the cited source actually supports the claim. A high value means the sentence really is grounded in its source.
- Grounding score (GS). Cosine similarity between the whole answer and the concatenation of all cited chunks. A single number for how close the answer stays to its evidence overall.

WE-DO here. One line: the evidence sufficiency cosine between a sentence and its cited chunk.

In [ ]:
def _answer_sentences(answer):
    return [s for s in re.split(r"(?<=\])\s+(?=[A-Z])", answer) if s.strip()]

CITE_RE = re.compile(r"\[[^\]]+:\d+\]\s*$")

def _chunk_text(doc_id, cid):
    for ch in chunks:
        if ch.doc_id == doc_id and ch.chunk_id == cid:
            return ch.text
    return None

def attribution_precision(answer):
    sents = _answer_sentences(answer)
    if not sents:
        return 0.0
    return sum(1 for s in sents if CITE_RE.search(s.strip())) / len(sents)

def evidence_sufficiency(answer):
    sents = _answer_sentences(answer)
    sims = []
    for s in sents:
        m = re.search(r"\[([^\]]+):(\d+)\]\s*$", s.strip())
        ct = _chunk_text(m.group(1), int(m.group(2))) if m else None
        if not ct:
            sims.append(0.0)
            continue
        bare = re.sub(r"\s*\[[^\]]+:\d+\]\s*$", "", s.strip())
        v = vectorizer.transform([bare, ct])
        # ===================== WE-DO =====================
        # Evidence sufficiency for this sentence: cosine similarity between the
        # bare sentence vector v[0] and its cited chunk vector v[1]. Ravel and
        # take element 0, cast to float, append to sims.
        sims.append(...)     # replace ... together
        # =================================================
    return float(np.mean(sims)) if sims else 0.0

def grounding_score(answer, cited_ids):
    texts = [t for t in (_chunk_text(d, c) for d, c in cited_ids) if t]
    base = " ".join(texts)
    if not base.strip() or answer.strip() in ("", "INSUFFICIENT_INFORMATION"):
        return 0.0
    bare = re.sub(r"\[[^\]]+:\d+\]", "", answer)
    v = vectorizer.transform([bare, base])
    return float(cosine_similarity(v[0], v[1]).ravel()[0])

def report(out):
    a = out["answer"]
    if a == "INSUFFICIENT_INFORMATION":
        return "abstained"
    return f"AP={attribution_precision(a):.2f} ES={evidence_sufficiency(a):.3f} GS={grounding_score(a, out['cited_ids']):.3f}"

print(report(synthesize("What is the return window for most items?")))

## Part 6: Demo queries and charts

Four questions that exercise the whole pipeline. Predict each result with the room before running.

1. A plain policy question. Grounds and cites cleanly.
2. A drill torque question. The knowledge base holds two conflicting claims, an official 500 and a forum 900. Watch retrieval surface both.
3. A question about a product that does not exist. The grounded system pulls the document that says it is not real, instead of inventing specs.
4. An off-topic question. The best match is weak, so the system abstains.

In [ ]:
DEMO_QUERIES = [
    "What is the return window for most items?",
    "How much torque does the Timberline 18 volt drill have?",
    "Tell me about the Cordwell SkyHammer 9000.",
    "How do I file my federal taxes?",
]

def bar_retrieval(query, k=5):
    res = retrieve(query, k=k)
    labels = [f"{d}:{c}" for (_, (d, c), _) in res]
    scores = [s for (s, _, _) in res]
    plt.figure(figsize=(6, 3))
    plt.bar(range(len(scores)), scores)
    plt.axhline(SIM_THRESHOLD, color="red", linestyle="--", label=f"abstain threshold {SIM_THRESHOLD}")
    plt.xticks(range(len(scores)), labels, rotation=45, ha="right")
    plt.title(query)
    plt.ylabel("cosine similarity")
    plt.legend()
    plt.tight_layout()
    plt.show()

for q in DEMO_QUERIES:
    print("=" * 70)
    print("QUERY:", q)
    bar_retrieval(q, k=5)
    out = synthesize(q)
    print("best_sim:", round(out["best_sim"], 3))
    print("answer:", out["answer"])
    print("faithfulness:", report(out))

What to point at, query by query.

- Return window. Clean grounding. Best similarity around 0.26, both sentences cited to the returns policy, ES around 0.69, GS near 1.0. This is the happy path.
- Drill torque. Best similarity around 0.50. Retrieval surfaces the official 500 inch pound spec and the conflicting 900 forum claim side by side, each cited. The system did not resolve the conflict for you. It made the conflict visible and traceable. A human sees one citation is a product sheet and one is a forum post, and vets accordingly. That is the honest behavior.
- SkyHammer 9000. The system retrieves the document that says the product is not real and leads with it, rather than hallucinating a spec sheet for a product that does not exist. Grounding turned a hallucination trap into a correct decline.
- Federal taxes. Best similarity is 0.0. Nothing in the knowledge base is close, so the system abstains. This is the behavior that keeps a confident wrong answer from reaching a customer.

## Part 7: Mini evaluation, abstention correctness

A tiny evaluation set that checks the one behavior that matters most for reliability: does the system answer when it should and decline when it should. Two answerable questions and two out of scope questions, scored for whether abstention matched expectation.

In [ ]:
EVAL = [
    ("What is the return window for most items?", False),
    ("How long does home delivery take?", False),
    ("How do I file my federal taxes?", True),
    ("What is the capital of France?", True),
]

rows, correct = [], 0
for q, expect_abstain in EVAL:
    out = synthesize(q)
    abstained = out["answer"] == "INSUFFICIENT_INFORMATION"
    ok = abstained == expect_abstain
    correct += ok
    rows.append({"query": q, "best_sim": round(out["best_sim"], 3),
                 "expected_abstain": expect_abstain, "abstained": abstained, "correct": ok})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nabstention accuracy: {correct}/{len(EVAL)}")

All four match on this run. The threshold cleanly separates the answerable questions, whose best similarity is well above it, from the off topic ones, whose best similarity is 0.0. In a real system you would sweep the threshold on a larger labeled set and pick the point that balances answering enough questions against declining the risky ones.

## Part 8: Optional generative answer

Everything so far is deterministic and extractive. This section swaps in a local language model to write the answer, while keeping the same evidence and the same abstention rule. The prompt forces a citation on every sentence. This is where you can show the contrast: a generative model may write more fluent prose, but it can also drift from its sources, which is exactly what the faithfulness metrics catch.

Backends. This runs against either LM Studio or Ollama, whichever the machine has. Both expose an OpenAI compatible endpoint, so the calling code is identical and only the port and model differ. The mode is a config variable. It defaults to offline, which returns the extractive answer, so the demo has zero external dependencies unless you opt in.

To go live, set the environment variable BACKEND_MODE to lmstudio or ollama before starting Jupyter, and make sure the server is running with the model pulled. If the chosen server is down, the call raises a clear error rather than silently pretending.

CURRENCY FLAG. The default local model tag is gemma4. Confirm the exact tag the cohort machines have actually pulled, for both LM Studio and Ollama, before class, and set LOCAL_MODEL if it differs.

WE-DO here. One line: the OpenAI compatible chat call that both backends share.

In [ ]:
# offline (default), lmstudio, or ollama. Read from the environment so nothing is hard-coded.
BACKEND_MODE = os.environ.get("BACKEND_MODE", "offline")
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "gemma4")   # CURRENCY FLAG: confirm the pulled tag

BACKENDS = {
    "lmstudio": {"base_url": "http://localhost:1234/v1", "api_key": "lm-studio"},
    "ollama":   {"base_url": "http://localhost:11434/v1", "api_key": "ollama"},
}

PROMPT_TPL = """You are a careful Cordwell assistant. Answer the QUESTION using ONLY the SNIPPETS.
Put a citation in the form [doc_id:chunk_id] at the end of every sentence.
If the snippets are insufficient, answer exactly: INSUFFICIENT_INFORMATION

SNIPPETS:
{snips}

QUESTION: {q}

ANSWER:"""

def make_client():
    """Return an OpenAI-compatible client for the selected backend, or None when offline."""
    if BACKEND_MODE == "offline":
        return None
    if BACKEND_MODE not in BACKENDS:
        raise ValueError(f"Unknown BACKEND_MODE {BACKEND_MODE!r}. Use offline, lmstudio, or ollama.")
    from openai import OpenAI
    cfg = BACKENDS[BACKEND_MODE]
    return OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])

def generative_answer(query, k=4, threshold=SIM_THRESHOLD):
    retrieved = retrieve(query, k=k)
    best = retrieved[0][0] if retrieved else 0.0
    if best < threshold:
        return {"answer": "INSUFFICIENT_INFORMATION", "cited_ids": [], "retrieved": retrieved, "best_sim": best}
    cited = [m for (_, m, _) in retrieved]
    if BACKEND_MODE == "offline":
        # Deterministic fallback: reuse the extractive answer so the demo always runs.
        return synthesize(query, k=k, threshold=threshold)

    snips = "\n".join(f"[{d}:{c}] {t}" for (_, (d, c), t) in retrieved)
    prompt = PROMPT_TPL.format(snips=snips, q=query)
    client = make_client()
    try:
        # ===================== WE-DO =====================
        # The OpenAI-compatible chat call that both LM Studio and Ollama share.
        # Use client.chat.completions.create with model=LOCAL_MODEL, a single
        # user message carrying prompt, and temperature=0.2. Then read the text
        # from resp.choices[0].message.content and strip it.
        resp = ...          # replace ... together
        answer = ...        # replace ... together
        # =================================================
    except Exception as e:
        raise RuntimeError(
            f"Backend {BACKEND_MODE} at {BACKENDS[BACKEND_MODE]['base_url']} is not reachable. "
            f"Start the server and pull {LOCAL_MODEL}, or set BACKEND_MODE=offline. Original error: {e}"
        )
    return {"answer": answer, "cited_ids": cited, "retrieved": retrieved, "best_sim": best}

print(f"BACKEND_MODE={BACKEND_MODE}  LOCAL_MODEL={LOCAL_MODEL}")
gen = generative_answer("How much torque does the Timberline 18 volt drill have?")
print("answer:", gen["answer"])
if gen["answer"] != "INSUFFICIENT_INFORMATION":
    print("faithfulness:", report(gen))

## Recap

- Grounding reduces hallucination risk. Answers are built from retrieved evidence, and every sentence cites its source.
- Abstention is the key reliability behavior. When the best match is weak, decline instead of guessing.
- Citations give traceability, not truth. The conflicting torque claim showed that grounding surfaces a conflict for a human to vet. It does not resolve it.
- Faithfulness is measurable. Evidence sufficiency and the grounding score are unit tests for whether the answer stayed true to its sources.

Responsible AI takeaway. Traceability is not the same as correctness. A cited answer is auditable, which is a real improvement, but you still measure factuality separately when the stakes are high, and you still keep an abstention policy so the system can say it does not know.

Where this goes next. Week 5 swaps TF-IDF for dense embeddings and a real vector store, and the same faithfulness metrics carry over. A later module adds a verifier pass that checks each claim against its evidence with a stronger model than cosine similarity.

## Pre-flight self-check

Run this last, before class. It confirms the reliability behaviors actually hold on this build. In the student notebook it passes only after every WE-DO line is filled in.

In [ ]:
# 1. Grounding: a normal policy question answers and cites.
ret = synthesize("What is the return window for most items?")
assert ret["answer"] != "INSUFFICIENT_INFORMATION", "Expected an answer for a covered question."
assert attribution_precision(ret["answer"]) == 1.0, "Every answer sentence should carry a citation."

# 2. Abstention: an off-topic question declines.
off = synthesize("How do I file my federal taxes?")
assert off["answer"] == "INSUFFICIENT_INFORMATION", "Off-topic question should abstain."

# 3. Conflict is surfaced: the torque query retrieves both the official and the forum chunk.
docs_hit = {d for (_, (d, _), _) in retrieve("How much torque does the Timberline 18 volt drill have?", k=4)}
assert "drill_spec_official" in docs_hit and "drill_forum_conflicting" in docs_hit, \
    "Both the official and conflicting drill sources should be retrieved."

# 4. Evidence sufficiency is high for an extractive answer.
assert evidence_sufficiency(ret["answer"]) > 0.4, "Extractive answer should be well grounded in its cited chunk."

print("Pre-flight checks passed.")
print("Grounding answers, abstention declines, conflict is surfaced, evidence is sufficient.")